# Apache Polaris — Clean All Resources

**Date:** 2026-05-15

## ⚠️ Warning
This notebook deletes ALL Polaris resources except system entities (root, service_admin).

**Deletion order matters** — must follow dependency order:
```
1. Tables & Views      (inside namespaces)
2. Namespaces          (inside catalogs)
3. Catalog Roles       (inside catalogs)
4. Catalogs
5. Principals          (except root)
6. Principal Roles     (except service_admin)
```

In [43]:
# ============================================================
# Prerequisites
# uv add requests pandas
# ============================================================
import requests
import json
import pandas as pd
from IPython.display import display

POLARIS_URL = 'http://192.168.139.2:8181'
REALM       = 'POLARIS'

def get_token(client_id='root', client_secret='polaris-secret'):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={'grant_type': 'client_credentials', 'client_id': client_id,
              'client_secret': client_secret, 'scope': 'PRINCIPAL_ROLE:ALL'}
    )
    return r.json()['access_token']

def h(token):
    return {'Authorization': f'Bearer {token}', 'Polaris-Realm': REALM,
            'Content-Type': 'application/json'}

def delete(url, token, label):
    r = requests.delete(url, headers=h(token))
    icon = '✅' if r.status_code in [200, 204] else '⚠️ '
    print(f'  {icon} [{r.status_code}] {label}')
    return r.status_code

TOKEN = get_token()

# System entities — never delete
SKIP_PRINCIPALS     = {'root'}
SKIP_PRINCIPAL_ROLES = {'service_admin'}

print('✅ Connected as root')
print(f'   Skip principals:      {SKIP_PRINCIPALS}')
print(f'   Skip principal roles: {SKIP_PRINCIPAL_ROLES}')

✅ Connected as root
   Skip principals:      {'root'}
   Skip principal roles: {'service_admin'}


In [54]:
# ============================================================
# Preview — show what will be deleted
# ============================================================
print('=== Resources to be deleted ===\n')

catalogs       = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs',       headers=h(TOKEN)).json().get('catalogs', [])
principals     = requests.get(f'{POLARIS_URL}/api/management/v1/principals',     headers=h(TOKEN)).json().get('principals', [])
principal_roles = requests.get(f'{POLARIS_URL}/api/management/v1/principal-roles', headers=h(TOKEN)).json().get('roles', [])

print(f'Catalogs ({len(catalogs)}):')
for c in catalogs:
    cname = c['name']
    # Namespaces
    ns_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(TOKEN))
    ns_list = ns_r.json().get('namespaces', [])
    print(f'  📚 {cname}')
    # Catalog roles
    cr_r = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN))
    for cr in cr_r.json().get('roles', []):
        print(f'    🔑 catalog-role: {cr["name"]}')
    for ns in ns_list:
        ns_name = ns[0] if isinstance(ns, list) else ns
        print(f'    📁 namespace: {ns_name}')
        # Tables
        t_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables', headers=h(TOKEN))
        for t in t_r.json().get('identifiers', []):
            print(f'      📊 table: {t.get("name", t)}')
        # Views
        v_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views', headers=h(TOKEN))
        for v in v_r.json().get('identifiers', []):
            print(f'      👁️  view: {v.get("name", v)}')

print(f'\nPrincipals ({len([p for p in principals if p["name"] not in SKIP_PRINCIPALS])} to delete):')
for p in principals:
    skip = p['name'] in SKIP_PRINCIPALS
    print(f'  {"⏭️  skip" if skip else "👤 delete"}: {p["name"]}')

print(f'\nPrincipal Roles ({len([r for r in principal_roles if r["name"] not in SKIP_PRINCIPAL_ROLES])} to delete):')
for pr in principal_roles:
    skip = pr['name'] in SKIP_PRINCIPAL_ROLES
    print(f'  {"⏭️  skip" if skip else "👥 delete"}: {pr["name"]}')

print('\n⚠️  Run next cell to execute deletion')

=== Resources to be deleted ===

Catalogs (1):
  📚 purge-practice
    🔑 catalog-role: catalog_admin
    📁 namespace: polaris-cleanup-trash-ns
      👁️  view: stuck_practice-ns_practice-view
    📁 namespace: practice-ns

Principals (0 to delete):
  ⏭️  skip: root

Principal Roles (0 to delete):
  ⏭️  skip: service_admin

⚠️  Run next cell to execute deletion


In [55]:
# ============================================================
# Step 1 — Delete Tables and Views
# ============================================================
print('=== Step 1: Delete Tables and Views ===')

for c in catalogs:
    cname = c['name']
    ns_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(TOKEN))
    for ns in ns_r.json().get('namespaces', []):
        ns_name = ns[0] if isinstance(ns, list) else ns

        # Delete tables
        t_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables', headers=h(TOKEN))
        for t in t_r.json().get('identifiers', []):
            tname = t.get('name', str(t))
            delete(
                f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables/{tname}',
                TOKEN, f'table: {cname}.{ns_name}.{tname}'
            )

        # Delete views
        v_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views', headers=h(TOKEN))
        for v in v_r.json().get('identifiers', []):
            vname = v.get('name', str(v))
            delete(
                f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views/{vname}',
                TOKEN, f'view:  {cname}.{ns_name}.{vname}'
            )

print('\n✅ Step 1 complete')

=== Step 1: Delete Tables and Views ===
  ⚠️  [403] view:  purge-practice.polaris-cleanup-trash-ns.stuck_practice-ns_practice-view

✅ Step 1 complete


In [23]:
# ============================================================
# Step 2 — Delete Namespaces
# ============================================================
print('=== Step 2: Delete Namespaces ===')

for c in catalogs:
    cname = c['name']
    ns_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(TOKEN))
    for ns in ns_r.json().get('namespaces', []):
        ns_name = ns[0] if isinstance(ns, list) else ns
        delete(
            f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}',
            TOKEN, f'namespace: {cname}.{ns_name}'
        )

print('\n✅ Step 2 complete')

=== Step 2: Delete Namespaces ===
  ⚠️  [409] namespace: purge-practice.practice-ns

✅ Step 2 complete


In [24]:
# ============================================================
# Step 3 — Delete Catalog Roles
# ============================================================
print('=== Step 3: Delete Catalog Roles ===')

for c in catalogs:
    cname = c['name']
    cr_r = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN))
    for cr in cr_r.json().get('roles', []):
        crname = cr['name']
        if crname == 'catalog_admin':
            print(f'  ⏭️  skip: {cname}/{crname} (system role)')
            continue
        delete(
            f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{crname}',
            TOKEN, f'catalog-role: {cname}/{crname}'
        )

print('\n✅ Step 3 complete')

=== Step 3: Delete Catalog Roles ===
  ⏭️  skip: watchdog-catalog/catalog_admin (system role)
  ✅ [204] catalog-role: watchdog-catalog/watchdog-catalog-role
  ⏭️  skip: purge-practice/catalog_admin (system role)

✅ Step 3 complete


In [25]:
# ============================================================
# Step 4 — Delete Catalogs
# ============================================================
print('=== Step 4: Delete Catalogs ===')

for c in catalogs:
    cname = c['name']
    delete(
        f'{POLARIS_URL}/api/management/v1/catalogs/{cname}',
        TOKEN, f'catalog: {cname}'
    )

print('\n✅ Step 4 complete')

=== Step 4: Delete Catalogs ===
  ✅ [204] catalog: watchdog-catalog
  ⚠️  [400] catalog: purge-practice

✅ Step 4 complete


In [26]:
# ============================================================
# Step 5 — Delete Principals
# ============================================================
print('=== Step 5: Delete Principals ===')

for p in principals:
    pname = p['name']
    if pname in SKIP_PRINCIPALS:
        print(f'  ⏭️  skip: {pname}')
        continue
    delete(
        f'{POLARIS_URL}/api/management/v1/principals/{pname}',
        TOKEN, f'principal: {pname}'
    )

print('\n✅ Step 5 complete')

=== Step 5: Delete Principals ===
  ⏭️  skip: root
  ✅ [204] principal: watchdog-principal

✅ Step 5 complete


In [27]:
# ============================================================
# Step 6 — Delete Principal Roles
# ============================================================
print('=== Step 6: Delete Principal Roles ===')

for pr in principal_roles:
    prname = pr['name']
    if prname in SKIP_PRINCIPAL_ROLES:
        print(f'  ⏭️  skip: {prname}')
        continue
    delete(
        f'{POLARIS_URL}/api/management/v1/principal-roles/{prname}',
        TOKEN, f'principal-role: {prname}'
    )

print('\n✅ Step 6 complete')

=== Step 6: Delete Principal Roles ===
  ✅ [204] principal-role: watchdog-principal-role
  ⏭️  skip: service_admin

✅ Step 6 complete


In [28]:
# ============================================================
# Verify — confirm everything is clean
# ============================================================
print('=== Verification ===')

remaining = {
    'catalogs':       requests.get(f'{POLARIS_URL}/api/management/v1/catalogs',        headers=h(TOKEN)).json().get('catalogs', []),
    'principals':     requests.get(f'{POLARIS_URL}/api/management/v1/principals',      headers=h(TOKEN)).json().get('principals', []),
    'principal_roles': requests.get(f'{POLARIS_URL}/api/management/v1/principal-roles', headers=h(TOKEN)).json().get('roles', []),
}

all_clean = True
for resource, items in remaining.items():
    names = [i['name'] for i in items]
    icon = '✅' if len(names) == 0 or all(n in SKIP_PRINCIPALS | SKIP_PRINCIPAL_ROLES for n in names) else '⚠️ '
    if icon == '⚠️ ':
        all_clean = False
    print(f'  {icon} {resource}: {names}')

print()
if all_clean:
    print('✅ All resources cleaned — only system entities remain')
else:
    print('⚠️  Some resources remain — check above')

=== Verification ===
  ⚠️  catalogs: ['purge-practice']
  ✅ principals: ['root']
  ✅ principal_roles: ['service_admin']

⚠️  Some resources remain — check above
